# EMT EDA — Exploratory Data Analysis
Visual inspection of the generated EMT dataset.
Run **after** `00_EMT_DataGen.ipynb` or `python emt/generate_data.py`.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('../..'))
from config import EMT_DATA_DIR, EMT

import numpy as np
import h5py
import matplotlib.pyplot as plt

# Check files
for fname in ["emt_train.h5", "emt_test.h5", "sensitivity_matrix_A.npy"]:
    p = EMT_DATA_DIR / fname
    exists = "OK" if p.exists() else "MISSING"
    print(f"  {fname:35s} [{exists}]")


## 1. Load Dataset and Show Basic Statistics

In [ ]:
with h5py.File(EMT_DATA_DIR / "emt_train.h5", "r") as f:
    images = f["images"][:]
    meas   = f["measurements"][:]
    n_obj  = f["n_objects"][:]

print(f"Train images  : {images.shape}  range [{images.min():.3f}, {images.max():.3f}]")
print(f"Measurements  : {meas.shape}   range [{meas.min():.3f}, {meas.max():.3f}]")
print(f"Object counts : 1-obj={np.sum(n_obj==1)}  2-obj={np.sum(n_obj==2)}  3-obj={np.sum(n_obj==3)}")


## 2. Sample Phantoms by Object Count

In [ ]:
fig, axes = plt.subplots(3, 8, figsize=(20, 8))

for row, n in enumerate([1, 2, 3]):
    idxs = np.where(n_obj == n)[0][:8]
    for col, idx in enumerate(idxs):
        axes[row, col].imshow(images[idx], cmap="hot")
        axes[row, col].axis("off")
    axes[row, 0].set_ylabel(f"{n} object(s)", fontsize=11, rotation=90, labelpad=5)

plt.suptitle("EMT Phantoms by Object Count (conductivity contrast)", fontsize=14)
plt.tight_layout()
plt.savefig("emt_eda_phantoms.png", dpi=120, bbox_inches="tight")
plt.show()


## 3. Measurement Vector Analysis

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].plot(meas[:5].T, alpha=0.7)
axes[0].set_title("5 sample measurement vectors", fontsize=12)
axes[0].set_xlabel("Measurement index"); axes[0].set_ylabel("Voltage diff (noisy)")
axes[0].grid(alpha=0.3)

axes[1].hist(meas.flatten(), bins=100, color="steelblue", alpha=0.8)
axes[1].set_title("Measurement value distribution", fontsize=12)
axes[1].set_xlabel("Value"); axes[1].grid(alpha=0.3)

# Mean absolute value per measurement channel
axes[2].plot(np.abs(meas).mean(axis=0))
axes[2].set_title("Mean |meas| per channel", fontsize=12)
axes[2].set_xlabel("Channel"); axes[2].grid(alpha=0.3)

plt.suptitle("EMT Measurement Analysis", fontsize=14)
plt.tight_layout()
plt.savefig("emt_eda_measurements.png", dpi=120, bbox_inches="tight")
plt.show()


## 4. DataLoader Sanity Check

In [ ]:
sys.path.insert(0, os.path.abspath('..'))
from dataset import build_emt_loaders

train_loader, val_loader, test_loader = build_emt_loaders()
b_batch, gt_batch = next(iter(train_loader))

print(f"Measurement batch : {b_batch.shape}  range [{b_batch.min():.3f}, {b_batch.max():.3f}]")
print(f"Image batch       : {gt_batch.shape} range [{gt_batch.min():.3f}, {gt_batch.max():.3f}]")

fig, axes = plt.subplots(2, 4, figsize=(16, 8))
for col in range(4):
    axes[0, col].imshow(gt_batch[col, 0].numpy(), cmap="hot")
    axes[0, col].set_title(f"GT sample {col}", fontsize=10); axes[0, col].axis("off")
    axes[1, col].plot(b_batch[col].numpy())
    axes[1, col].set_title(f"Meas sample {col}", fontsize=10); axes[1, col].grid(alpha=0.3)

plt.suptitle("DataLoader Batch", fontsize=13)
plt.tight_layout()
plt.savefig("emt_eda_batch.png", dpi=120, bbox_inches="tight")
plt.show()


## 5. Sensitivity Matrix A — Singular Value Decomposition

In [ ]:
A = np.load(EMT_DATA_DIR / "sensitivity_matrix_A.npy")
U, S, Vt = np.linalg.svd(A, full_matrices=False)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].semilogy(S, "b-o", markersize=4)
axes[0].set_title("Singular Values of A", fontsize=12)
axes[0].set_xlabel("Index"); axes[0].set_ylabel("Singular value (log scale)")
axes[0].grid(alpha=0.3)

cumvar = np.cumsum(S**2) / np.sum(S**2)
axes[1].plot(cumvar, "r-")
axes[1].axhline(0.99, ls="--", c="gray", label="99% var")
axes[1].set_title("Cumulative Explained Variance", fontsize=12)
axes[1].set_xlabel("# components"); axes[1].legend(); axes[1].grid(alpha=0.3)

print(f"A shape: {A.shape}")
print(f"Condition number: {S[0]/S[-1]:.2e}")
print(f"Components for 99% variance: {np.searchsorted(cumvar, 0.99)+1}")

plt.tight_layout()
plt.savefig("emt_eda_svd.png", dpi=120, bbox_inches="tight")
plt.show()
